### Fase 1: Transformando Canciones en "Canastas"
El algoritmo Apriori no lee DataFrames normales; necesita una lista de transacciones. 
En nuestro caso, cada **Canción** es una transacción, y los **Artistas** que participan en ella son los artículos de la canasta.

In [ ]:
# Instalar si es necesario: !pip install mlxtend
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# 1. INGESTA DE DATOS
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/Most%20Streamed%20Spotify%20Songs%202024.csv"
df_spotify = pd.read_csv(url, encoding='latin-1')

# 2. LIMPIEZA: Eliminar nulos de la columna de artistas
df_spotify = df_spotify.dropna(subset=['Artist'])

# 3. TRANSFORMACIÓN: Separar los artistas que colaboran en cada canción
# Asumimos que las colaboraciones están separadas por coma (ej. "Bad Bunny, Feid")
# Convertimos esa columna en una verdadera lista de Python

df_spotify['Artist'] = df_spotify['Artist'].str.replace('Tyler, The Creator', 'Tyler The Creator')
invitados = df_spotify['Track'].str.split('(feat. ', regex=False).str[1].str.replace(')', '', regex=False).fillna('')
lista_colaboraciones = (df_spotify['Artist'] + ', ' + invitados).str.split(', ').tolist()

# Muestra rápida de cómo se ven nuestras "canastas"
print("Muestra de las primeras 5 colaboraciones:")
for i in range(5):
    print(f"Canción {i+1}: {lista_colaboraciones[i]}")


Muestra de las primeras 5 colaboraciones:
Canción 1: ['Tommy Richman', '']
Canción 2: ['Kendrick Lamar', '']
Canción 3: ['Artemas', '']
Canción 4: ['Miley Cyrus', '']
Canción 5: ['Eminem', '']


In [16]:
# Utiliza TransactionEncoder para convertir 'lista_colaboraciones' en una matriz booleana (One-Hot)
# Guárdala en un DataFrame llamado 'df_transacciones'

te = TransactionEncoder()
matriz_bool = te.fit_transform(lista_colaboraciones)
df_transacciones = pd.DataFrame(matriz_bool, columns=te.columns_)

### Fase 2: Aplicando Apriori
Ahora que tenemos la matriz con ceros y unos, extraigamos los conjuntos de artistas que colaboran frecuentemente.
Debido a que hay miles de artistas solistas, las colaboraciones son estadísticamente raras. Usaremos un `min_support` muy bajo para capturar las colaboraciones.

In [18]:
# --- RETO PARA EL ALUMNO AQUÍ ---

# 1. Encuentra los Itemsets Frecuentes usando un soporte mínimo del 0.005 (0.5% de las canciones)
itemsets_frecuentes = apriori(df_transacciones, min_support=0.0004, use_colnames=True)

# 2. Genera las Reglas de Asociación filtrando por la métrica 'lift' con un mínimo de 1.2
reglas_musicales = association_rules(itemsets_frecuentes, metric="lift", min_threshold=1.2)

# 3. Muestra las reglas resultantes ordenadas por 'confidence' de mayor a menor
display(reglas_musicales.sort_values(by="confidence", ascending=False))

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
47,frozenset({Norma Jean Martine}),frozenset({Ofenbach}),0.000435,0.000653,0.000435,1.000000,1531.666667,1.0,0.000435,inf,0.999782,0.666667,1.000000,0.833333
36,frozenset({Nicki Minaj & Ty Dolla $ign}),frozenset({Jason Derulo}),0.000435,0.002394,0.000435,1.000000,417.727273,1.0,0.000434,inf,0.998040,0.181818,1.000000,0.590909
17,frozenset({L.V.}),frozenset({Coolio}),0.000435,0.000653,0.000435,1.000000,1531.666667,1.0,0.000435,inf,0.999782,0.666667,1.000000,0.833333
27,frozenset({Sexyy Red & SZA}),frozenset({Drake}),0.000435,0.017845,0.000435,1.000000,56.036585,1.0,0.000427,inf,0.982582,0.024390,1.000000,0.512195
25,frozenset({PARTYNEXTDOOR}),frozenset({Drake}),0.000871,0.017845,0.000653,0.750000,42.027439,1.0,0.000637,3.928618,0.977057,0.036145,0.745458,0.393293
3,frozenset({42 Dugg}),frozenset({Lil Baby}),0.000653,0.005441,0.000435,0.666667,122.533333,1.0,0.000432,2.983678,0.992487,0.076923,0.664843,0.373333
46,frozenset({Ofenbach}),frozenset({Norma Jean Martine}),0.000653,0.000435,0.000435,0.666667,1531.666667,1.0,0.000435,2.998694,1.000000,0.666667,0.666522,0.833333
7,frozenset({Snoop Dogg}),frozenset({Bruno Mars}),0.000653,0.003047,0.000435,0.666667,218.809524,1.0,0.000433,2.990860,0.996080,0.133333,0.665648,0.404762
16,frozenset({Coolio}),frozenset({L.V.}),0.000653,0.000435,0.000435,0.666667,1531.666667,1.0,0.000435,2.998694,1.000000,0.666667,0.666522,0.833333
14,frozenset({Wiz Khalifa}),frozenset({Charlie Puth}),0.000871,0.001523,0.000435,0.500000,328.214286,1.0,0.000434,1.996953,0.997822,0.222222,0.499237,0.392857


### 🧠 Fase 3: Interpretación de Negocio (El Entregable)

Como Científico de Datos, no basta con entregar una tabla. Debes explicarle los hallazgos al Director Creativo. Revisa tu DataFrame `reglas_musicales` y responde las siguientes 4 preguntas:

**1. Análisis de Soporte (El Volumen):**
Menciona a los dos artistas que conforman la regla con el mayor **Soporte**. ¿Qué significa este número en términos simples para el negocio?

> **Tu Respuesta:** Los artistas son Drake y 21 Savage con un soporte de 0.00065, lo que significa que este dúo colabora en 3 canciones del top global de 2024.

**2. Análisis de Confianza (La Dirección):**
Busca una regla que tenga una **Confianza** mayor al 0.5 (50%). Traduce esta regla matemática a una frase que el director pueda entender (Ej. "Si contratamos al artista A, existe un X% de probabilidad de que el artista B también deba estar en el track para que sea el éxito que registramos").

> **Tu Respuesta:** Por ejemplo la regla 42 Dugg a Lil Baby tiene una confianza de 0.67, lo que podría significar que si contratamos a 42 Dugg para una canción, existe un 66.7% de probabilidad de que Lil Baby también deba participar como colaborador para que la canción alcance el nivel de éxito que tenemos registrado en el top.

**3. Análisis de Lift (El Impulso):**
Busca la regla con el **Lift** más alto de toda tu tabla. 
¿Por qué esta métrica es más confiable que el Soporte para decidir si estos dos artistas *realmente* tienen química musical o si solo fue una coincidencia estadística?

> **Tu Respuesta:** La regla de 42 Dugg a Lil Baby tienen un lift muy alto (122.5).
Esto es más confiable que el soporte porque el soporte solo cuenta cuántas canciones tienen, mientras que el lift elimina la casualidad.

**4. Toma de Decisión:**
Basado puramente en las reglas matemáticas que descubriste, si la disquera solo tiene presupuesto para juntar a dos artistas en el estudio la próxima semana, ¿qué pareja elegirías y por qué? (Justifica usando las 3 métricas de oro).

> **Tu Respuesta:** Elegiría juntar a Drake y 21 Savage porque tienen el soporte más alto del ranking, demostrando que el público consume su contenido, tienen una buena confianza lo que dice que si 21 Savage lanza un tema, hay una confianza considerable de que Drake impulse la pista al éxito y su Lift es de 7.0, lo que confirma que su unión multiplica 7 veces la probabilidad de éxito.